In [1]:
using PowerModels, Ipopt, MathOptInterface, ProgressMeter, DelimitedFiles, Random, Statistics, DataFrames, CSV
const MOI = MathOptInterface
Random.seed!(42) 

function solve_ACOPF(case_path::String, num_samples::Int=50000, variance::Float64=0.12)
    println("Samples: $(num_samples), Variance: $(variance)")
    
    data = parse_file(case_path)
    
    base_loads_pd = Dict(load["load_bus"] => load["pd"] for (load_idx, load) in data["load"] if load["pd"] > 0)
    base_loads_qd = Dict(load["load_bus"] => load["qd"] for (load_idx, load) in data["load"] if haskey(load, "qd"))

    load_bus_indices = sort(collect(keys(base_loads_pd)))
    gen_indices = sort([g["index"] for (i, g) in data["gen"]])
    bus_indices = sort([b["bus_i"] for (i,b) in data["bus"]])

    silent_optimizer = MOI.OptimizerWithAttributes(Ipopt.Optimizer, "print_level" => 0, "sb" => "yes")
    
    successful_samples = 0
    load_pd_list = Vector{Vector{Float64}}()
    load_qd_list = Vector{Vector{Float64}}() 
    gen_pg_list = Vector{Vector{Float64}}()
    gen_qg_list = Vector{Vector{Float64}}()
    bus_vm_list = Vector{Vector{Float64}}()
    bus_va_list = Vector{Vector{Float64}}()
    cost_list = Float64[]

    total_loop_time = @elapsed begin
        @showprogress "Progress" for _ in 1:num_samples 
            temp_data = deepcopy(data)

            new_loads_pd = Dict{Int, Float64}()
            new_loads_qd = Dict{Int, Float64}()
            for bus_idx in load_bus_indices
                base_pd = base_loads_pd[bus_idx]
                sigma_pd = abs(base_pd * variance)
                new_loads_pd[bus_idx] = max(0.0, base_pd + sigma_pd * randn())
                
                base_qd = get(base_loads_qd, bus_idx, 0.0)
                sigma_qd = abs(base_qd * variance) 
                new_loads_qd[bus_idx] = base_qd + sigma_qd * randn()
            end

            for (load_id, load_data) in temp_data["load"]
                bus_idx_int = load_data["load_bus"]
                if haskey(new_loads_pd, bus_idx_int)
                    load_data["pd"] = new_loads_pd[bus_idx_int]
                    load_data["qd"] = new_loads_qd[bus_idx_int] 
                end
            end

            try
                result = solve_ac_opf(temp_data, silent_optimizer; setting = Dict("output" => Dict("branch_flows" => true)))

                if result["termination_status"] == MOI.LOCALLY_SOLVED
                    push!(cost_list, result["objective"])
                    
                    push!(load_pd_list, [new_loads_pd[i] for i in load_bus_indices])
                    push!(load_qd_list, [new_loads_qd[i] for i in load_bus_indices])
                    
                    push!(gen_pg_list, [result["solution"]["gen"][string(i)]["pg"] for i in gen_indices])
                    push!(gen_qg_list, [result["solution"]["gen"][string(i)]["qg"] for i in gen_indices])
                    push!(bus_vm_list, [result["solution"]["bus"][string(i)]["vm"] for i in bus_indices])
                    push!(bus_va_list, [result["solution"]["bus"][string(i)]["va"] for i in bus_indices])
                    
                    successful_samples += 1
                end
            catch e
            end
        end 
    end 
    
    println("Successfully generated $(successful_samples) / $(num_samples) samples in $(round(total_loop_time, digits=2))s.")
    
    if !isempty(cost_list)
        avg_cost = mean(cost_list)
        println("Average cost: ", round(avg_cost, digits=2),"", string('$'), "/h")
    end
    
    if successful_samples > 0
        avg_solve_time_ms = (total_loop_time / successful_samples) * 1000 
        println("Average solving time for each sample: ", round(avg_solve_time_ms, digits=2),"ms")
    end
    
    output_dir = raw"C:\Users\Aloha\Desktop\dataset\ACOPF dataset\case118(api)" #change the output address
    mkpath(output_dir)
    case_name = split(basename(case_path), ".")[1]

    function save_to_csv(data_list, col_indices, type, name)
        if !isempty(data_list)
            matrix = hcat(data_list...)'
            df = DataFrame(matrix, Symbol.(type .* string.(col_indices)))
            filepath = joinpath(output_dir, "$(case_name)_$(name).csv")
            CSV.write(filepath, df)
        end
    end

    save_to_csv(load_pd_list, load_bus_indices, "pd", "pd")
    save_to_csv(load_qd_list, load_bus_indices, "qd", "qd") 
    save_to_csv(gen_pg_list, gen_indices, "pg_", "pg")
    save_to_csv(gen_qg_list, gen_indices, "qg_", "qg")
    save_to_csv(bus_vm_list, bus_indices, "vm_", "vm")
    save_to_csv(bus_va_list, bus_indices, "va_", "va")
end

# Main
CASE_FILE_PATH = raw"C:\Users\Aloha\Desktop\dataset\PGlib\api\pglib_opf_case118_ieee__api.m"
solve_ACOPF(CASE_FILE_PATH, 50000, 0.12) 

Samples: 50000, Variance: 0.12


SystemError: SystemError: opening file "C:\\Users\\Aloha\\Desktop\\dataset\\PGlib\\api\\pglib_opf_case118_ieee__api.m": No such file or directory

In [2]:
using PowerModels, Ipopt, MathOptInterface, ProgressMeter, DelimitedFiles, Random, Statistics, DataFrames, CSV

const MOI = MathOptInterface

"""
Generates an AC-OPF dataset by solving multiple power flow problems
with stochastically sampled loads. Supports both uniform and Gaussian sampling methods.
"""
function solve_ACOPF(case_path::String; 
                     num_samples::Int=5000, 
                     variance::Float64=0.12, 
                     sampling_method::String="uniform", 
                     sampling_range::Vector{Float64}=[0.9, 1.1],
                     output_dir::String)
    
    # --- 1. Initialization ---
    Random.seed!(42)
    data = parse_file(case_path)
    case_name = split(basename(case_path), ".")[1]

    # Print configuration
    println("Generating AC-OPF dataset for: $(case_name)")
    println("Target samples: $(num_samples), Sampling method: $(sampling_method)")
    if sampling_method == "uniform"
        println("Sampling range: $(sampling_range[1]*100)% to $(sampling_range[2]*100)%")
    else
        println("Sampling variance: $(variance)")
    end

    # Extract base load and network topology information
    base_loads_pd = Dict(load["load_bus"] => load["pd"] for (load_idx, load) in data["load"] if load["pd"] > 0)
    base_loads_qd = Dict(load["load_bus"] => load["qd"] for (load_idx, load) in data["load"] if haskey(load, "qd"))

    load_bus_indices = sort(collect(keys(base_loads_pd)))
    gen_indices = sort([g["index"] for (i, g) in data["gen"]])
    bus_indices = sort([b["bus_i"] for (i,b) in data["bus"]])

    # Configure the optimizer to run silently
    silent_optimizer = MOI.OptimizerWithAttributes(Ipopt.Optimizer, "print_level" => 0, "sb" => "yes")
    
    # Initialize lists to store results from successful simulations
    successful_samples = 0
    load_pd_list = Vector{Vector{Float64}}()
    load_qd_list = Vector{Vector{Float64}}() 
    gen_pg_list = Vector{Vector{Float64}}()
    gen_qg_list = Vector{Vector{Float64}}()
    bus_vm_list = Vector{Vector{Float64}}()
    bus_va_list = Vector{Vector{Float64}}()
    cost_list = Float64[]

    # --- 2. Sampling and Solving Loop ---
    total_loop_time = @elapsed begin
        @showprogress "Generating Samples" for _ in 1:num_samples 
            temp_data = deepcopy(data)
            new_loads_pd = Dict{Int, Float64}()
            new_loads_qd = Dict{Int, Float64}()

            # Apply sampling to each load bus
            for bus_idx in load_bus_indices
                base_pd = base_loads_pd[bus_idx]
                base_qd = get(base_loads_qd, bus_idx, 0.0)
                
                local new_pd, new_qd

                if sampling_method == "gaussian"
                    sigma_pd = abs(base_pd * variance)
                    new_pd = max(0.0, base_pd + sigma_pd * randn())
                    
                    sigma_qd = abs(base_qd * variance) 
                    new_qd = base_qd + sigma_qd * randn()

                elseif sampling_method == "uniform"
                    # Sample active power (pd)
                    lower_pd = base_pd * sampling_range[1]
                    upper_pd = base_pd * sampling_range[2]
                    new_pd = lower_pd + (upper_pd - lower_pd) * rand()
                    
                    # Sample reactive power (qd)
                    lower_qd = base_qd * sampling_range[1]
                    upper_qd = base_qd * sampling_range[2]
                    # Ensure correct bounds if base_qd is negative
                    if lower_qd > upper_qd
                        lower_qd, upper_qd = upper_qd, lower_qd
                    end
                    new_qd = lower_qd + (upper_qd - lower_qd) * rand()
                else
                    error("Unsupported sampling method: $(sampling_method). Use 'gaussian' or 'uniform'.")
                end
                
                new_loads_pd[bus_idx] = new_pd
                new_loads_qd[bus_idx] = new_qd
            end

            # Update the temporary data object with new loads
            for (load_id, load_data) in temp_data["load"]
                bus_idx_int = load_data["load_bus"]
                if haskey(new_loads_pd, bus_idx_int)
                    load_data["pd"] = new_loads_pd[bus_idx_int]
                    load_data["qd"] = new_loads_qd[bus_idx_int] 
                end
            end

            # Solve the AC-OPF problem
            try
                result = solve_ac_opf(temp_data, silent_optimizer; setting = Dict("output" => Dict("branch_flows" => true)))

                if result["termination_status"] == MOI.LOCALLY_SOLVED
                    successful_samples += 1
                    push!(cost_list, result["objective"])
                    
                    # Store results if solver was successful
                    push!(load_pd_list, [new_loads_pd[i] for i in load_bus_indices])
                    push!(load_qd_list, [new_loads_qd[i] for i in load_bus_indices])
                    
                    push!(gen_pg_list, [result["solution"]["gen"][string(i)]["pg"] for i in gen_indices])
                    push!(gen_qg_list, [result["solution"]["gen"][string(i)]["qg"] for i in gen_indices])
                    push!(bus_vm_list, [result["solution"]["bus"][string(i)]["vm"] for i in bus_indices])
                    push!(bus_va_list, [result["solution"]["bus"][string(i)]["va"] for i in bus_indices])
                end
            catch e
                # Silently ignore solver errors and continue to the next sample
            end
        end 
    end 
    
    # --- 3. Reporting Statistics ---
    println("Successfully generated $(successful_samples) / $(num_samples) samples in $(round(total_loop_time, digits=2))s.")
    
    if !isempty(cost_list)
        avg_cost = mean(cost_list)
        println("Average cost: ", round(avg_cost, digits=2), "\$/h")
    end
    
    if successful_samples > 0
        avg_solve_time_ms = (total_loop_time / successful_samples) * 1000 
        println("Average solving time for each sample: ", round(avg_solve_time_ms, digits=2),"ms")
    end
    
    # --- 4. Saving Data to CSV ---
    mkpath(output_dir)

    # Helper function to convert data lists to DataFrames and save as CSV
    function save_to_csv(data_list, col_indices, type_prefix, file_suffix)
        if !isempty(data_list)
            matrix = hcat(data_list...)'
            # Use underscores for better readability, e.g., "pd_1", "pg_1"
            df = DataFrame(matrix, Symbol.(type_prefix .* "_" .* string.(col_indices)))
            filepath = joinpath(output_dir, "$(case_name)_$(file_suffix).csv")
            CSV.write(filepath, df)
            println("Saved data to $(filepath)")
        end
    end

    save_to_csv(load_pd_list, load_bus_indices, "pd", "loads_pd")
    save_to_csv(load_qd_list, load_bus_indices, "qd", "loads_qd") 
    save_to_csv(gen_pg_list, gen_indices, "pg", "gens_pg")
    save_to_csv(gen_qg_list, gen_indices, "qg", "gens_qg")
    save_to_csv(bus_vm_list, bus_indices, "vm", "buses_vm")
    save_to_csv(bus_va_list, bus_indices, "va", "buses_va")
end

function main()
    # --- CONFIGURATION ---
    # Define root directory and case file
    ROOT_DIR = raw"C:\Users\Aloha\Desktop\dataset"
    CASE_FILE_NAME = "pglib_opf_case57_ieee.m"
    CASE_NAME_SHORT = "case57" # For output folder naming

    CASE_FILE_PATH = joinpath(ROOT_DIR, "PGlib", "standard", CASE_FILE_NAME)

    # Choose sampling method: "uniform" or "gaussian"
    SAMPLING_METHOD = "uniform"
    
    # Settings for uniform sampling
    SAMPLING_RANGE = [0.9, 1.1] # [90%, 110%] range
    
    # Settings for Gaussian sampling (used only if SAMPLING_METHOD = "gaussian")
    VARIANCE = 0.12

    NUM_SAMPLES = 50000 # Number of samples to generate
    
    # --- Generate a descriptive label for the output folder ---
    local sampling_label
    if SAMPLING_METHOD == "uniform"
        lower_pct = round(Int, SAMPLING_RANGE[1] * 100)
        upper_pct = round(Int, SAMPLING_RANGE[2] * 100)
        sampling_label = "u=$(lower_pct)-$(upper_pct)"
    else # Gaussian
        sampling_label = "v=$(VARIANCE)"
    end
    
    # Define the output directory path
    output_dir = joinpath(ROOT_DIR, "ACOPF dataset", "$(CASE_NAME_SHORT)($(sampling_label))")
    
    # --- Run the main function ---
    solve_ACOPF(
        CASE_FILE_PATH, 
        num_samples=NUM_SAMPLES, 
        variance=VARIANCE,
        sampling_method=SAMPLING_METHOD,
        sampling_range=SAMPLING_RANGE,
        output_dir=output_dir
    )
end

# --- Execute the script ---
main()

[info | PowerModels]: removing 3 cost terms from generator 4: Float64[]
[info | PowerModels]: removing 1 cost terms from generator 1: [1696.0624, 0.0]
[info | PowerModels]: removing 1 cost terms from generator 5: [3044.1037, 0.0]
[info | PowerModels]: removing 3 cost terms from generator 2: Float64[]
[info | PowerModels]: removing 3 cost terms from generator 6: Float64[]
[info | PowerModels]: removing 1 cost terms from generator 7: [3718.8979000000004, 0.0]
[info | PowerModels]: removing 1 cost terms from generator 3: [3407.5557000000003, 0.0]
Generating AC-OPF dataset for: pglib_opf_case57_ieee
Target samples: 50000, Sampling method: uniform
Sampling range: 90.0% to 110.00000000000001%


Generating Samples 100%|█████████████████████████████████| Time: 0:54:44


Successfully generated 50000 / 50000 samples in 3284.03s.
Average cost: 37594.47$/h
Average solving time for each sample: 65.68ms
Saved data to C:\Users\Aloha\Desktop\dataset\ACOPF dataset\case57(u=90-110)\pglib_opf_case57_ieee_loads_pd.csv
Saved data to C:\Users\Aloha\Desktop\dataset\ACOPF dataset\case57(u=90-110)\pglib_opf_case57_ieee_loads_qd.csv
Saved data to C:\Users\Aloha\Desktop\dataset\ACOPF dataset\case57(u=90-110)\pglib_opf_case57_ieee_gens_pg.csv
Saved data to C:\Users\Aloha\Desktop\dataset\ACOPF dataset\case57(u=90-110)\pglib_opf_case57_ieee_gens_qg.csv
Saved data to C:\Users\Aloha\Desktop\dataset\ACOPF dataset\case57(u=90-110)\pglib_opf_case57_ieee_buses_vm.csv
Saved data to C:\Users\Aloha\Desktop\dataset\ACOPF dataset\case57(u=90-110)\pglib_opf_case57_ieee_buses_va.csv
